## Data Cube Construction and OLAP Operations

Online Analytical Processing (OLAP) is a technology used in data analysis to explore large datasets from multiple perspectives. It allows users to perform complex queries and extract meaningful insights from multidimensional data.

A **Data Cube** is a multidimensional representation of data where each dimension represents a different attribute such as time, location, product category, or customer information. OLAP operations allow analysts to navigate, summarize, and analyze this multidimensional data efficiently.

In this notebook, we construct a data cube and perform several OLAP operations on the shopping behavior dataset to analyze purchasing patterns and trends.

The following OLAP operations are demonstrated:

- **Roll-up** – Aggregating data to a higher level of summarization.
- **Drill-down** – Breaking aggregated data into more detailed levels.
- **Slice** – Selecting a single value for one dimension to analyze a specific subset of data.
- **Dice** – Selecting multiple values across different dimensions to create a smaller sub-cube of the dataset.
- **Pivot (or Rotate)** – Reorienting the data cube by changing the dimensional perspective of the data, often using pivot tables to reorganize rows and columns.

These operations allow analysts to examine the data at different levels of granularity, uncover patterns, and gain deeper insights into customer purchasing behavior.

In [3]:
import pandas as pd
df = pd.read_csv("shopping_behavior_updated.csv")
df.head()


,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [4]:
df.columns

Index(['Customer ID', 'Age', 'Gender', 'Item Purchased', 'Category',
       'Purchase Amount (USD)', 'Location', 'Size', 'Color', 'Season',
       'Review Rating', 'Subscription Status', 'Shipping Type',
       'Discount Applied', 'Promo Code Used', 'Previous Purchases',
       'Payment Method', 'Frequency of Purchases'],
      dtype='object')

## Pivot Operation

The **Pivot** operation reorganizes the data by rotating the data cube to view it from a different perspective.

Here, a pivot table is used to summarize purchase amounts across different dimensions such as **Season and Category**, making comparisons easier.

In [5]:
data_cube = pd.pivot_table(
    df,
    values="Purchase Amount (USD)",
    index="Category",
    columns="Season",
    aggfunc="sum"
)

print("Data Cube (Pivot Table):")
print(data_cube)

Data Cube (Pivot Table):
Season        Fall  Spring  Summer  Winter
Category                                  
Accessories  19874   17007   19028   18291
Clothing     26220   27692   23078   27274
Footwear      8665    9555    9393    8480
Outerwear     5259    4425    4278    4562


## Roll-Up Operation

The **Roll-up** operation aggregates detailed data into a higher level of summarization.

In this analysis, two roll-up operations are performed:
- First, the total **Purchase Amount** is aggregated by **Category** to observe overall category-level sales.
- Second, the data is aggregated by **Item Purchased** to examine the total sales for each individual product.

These roll-ups provide a summarized view of purchasing patterns at different levels of detail.

In [6]:
# Roll-up by Item category
rollup = df.groupby("Category")["Purchase Amount (USD)"].sum()

print("Roll-Up Result (Total Purchase per Category):")
print(rollup)

Roll-Up Result (Total Purchase per Category):
Category
Accessories     74200
Clothing       104264
Footwear        36093
Outerwear       18524
Name: Purchase Amount (USD), dtype: int64


In [7]:
# Roll-up by Item Purchased
rollup_items = df.groupby("Item Purchased")["Purchase Amount (USD)"].sum()

print("Roll-Up Result (Total Purchase per Item):")
print(rollup_items)


Roll-Up Result (Total Purchase per Item):
Item Purchased
Backpack       8636
Belt           9635
Blouse        10410
Boots          9018
Coat           9275
Dress         10320
Gloves         8477
Handbag        8857
Hat            9375
Hoodie         8767
Jacket         9249
Jeans          7548
Jewelry       10010
Pants         10090
Sandals        9200
Scarf          9561
Shirt         10332
Shoes          9240
Shorts         9433
Skirt          9402
Sneakers       8635
Socks          9252
Sunglasses     9649
Sweater        9462
T-shirt        9248
Name: Purchase Amount (USD), dtype: int64


## Drill-Down Operation

The **Drill-down** operation moves from summarized data to a more detailed level of analysis.

In this notebook, two drill-down analyses are performed:
- From **Category to Item Purchased** to understand which items contribute to each category's sales.
- From **Season to Item Purchased** to observe which products are purchased during different seasons.

This helps reveal more detailed patterns within the aggregated data.

In [8]:
#drill down item purchased by category
drilldown = df.groupby(["Category", "Item Purchased"])["Purchase Amount (USD)"].sum()

print("Drill-Down Result (Category → Item Purchased):")
print(drilldown)

Drill-Down Result (Category → Item Purchased):
Category     Item Purchased
Accessories  Backpack           8636
             Belt               9635
             Gloves             8477
             Handbag            8857
             Hat                9375
             Jewelry           10010
             Scarf              9561
             Sunglasses         9649
Clothing     Blouse            10410
             Dress             10320
             Hoodie             8767
             Jeans              7548
             Pants             10090
             Shirt             10332
             Shorts             9433
             Skirt              9402
             Socks              9252
             Sweater            9462
             T-shirt            9248
Footwear     Boots              9018
             Sandals            9200
             Shoes              9240
             Sneakers           8635
Outerwear    Coat               9275
             Jacket             9249


In [9]:
#drill down item purchased by season
drilldown_season_item = df.groupby(["Season", "Item Purchased"])["Purchase Amount (USD)"].sum()

print("Drill-Down Result (Season → Item Purchased):")
print(drilldown_season_item)


Drill-Down Result (Season → Item Purchased):
Season  Item Purchased
Fall    Backpack          2008
        Belt              2553
        Blouse            2690
        Boots             2159
        Coat              2153
                          ... 
Winter  Sneakers          2220
        Socks             2170
        Sunglasses        3085
        Sweater           2365
        T-shirt           2511
Name: Purchase Amount (USD), Length: 100, dtype: int64


## Slice Operation

The **Slice** operation extracts a subset of the dataset by selecting a single value from one dimension.

Two slices are created in this analysis:
- Transactions where **Season = Summer**
- Transactions where **Season = Winter**

These slices allow us to focus on and analyze purchase behavior within specific seasons.

In [10]:
#slice 
slice_data = df[df["Season"] == "Summer"]

print("Slice Result (Summer Purchases):")
print(slice_data.head())

Slice Result (Summer Purchases):
    Customer ID  Age Gender Item Purchased   Category  Purchase Amount (USD)  \
5             6   46   Male       Sneakers   Footwear                     20   
8             9   26   Male           Coat  Outerwear                     97   
18           19   52   Male        Sweater   Clothing                     48   
19           20   66   Male          Pants   Clothing                     90   
22           23   56   Male          Pants   Clothing                     37   

         Location Size   Color  Season  Review Rating Subscription Status  \
5         Wyoming    M   White  Summer            2.9                 Yes   
8   West Virginia    L  Silver  Summer            2.6                 Yes   
18        Montana    S   Black  Summer            4.6                 Yes   
19   Rhode Island    M   Green  Summer            3.3                 Yes   
22     California    M   Peach  Summer            3.2                 Yes   

    Shipping Type Disco

In [11]:
#slice
slice_data = df[df["Season"] == "Winter"]

print("Slice Result (Winter Purchases):")
print(slice_data.head())

Slice Result (Winter Purchases):
    Customer ID  Age Gender Item Purchased   Category  Purchase Amount (USD)  \
0             1   55   Male         Blouse   Clothing                     53   
1             2   19   Male        Sweater   Clothing                     64   
7             8   27   Male         Shorts   Clothing                     34   
11           12   30   Male         Shorts   Clothing                     68   
12           13   61   Male           Coat  Outerwear                     72   

     Location Size     Color  Season  Review Rating Subscription Status  \
0    Kentucky    L      Gray  Winter            3.1                 Yes   
1       Maine    L    Maroon  Winter            3.1                 Yes   
7   Louisiana    L  Charcoal  Winter            3.2                 Yes   
11     Hawaii    S     Olive  Winter            4.9                 Yes   
12   Delaware    M      Gold  Winter            4.5                 Yes   

    Shipping Type Discount Applied 

In [12]:
slice_summary = slice_data.groupby("Category")["Purchase Amount (USD)"].sum()
print(slice_summary)

Category
Accessories    18291
Clothing       27274
Footwear        8480
Outerwear       4562
Name: Purchase Amount (USD), dtype: int64


## Dice Operation

The **Dice** operation filters the dataset using multiple conditions across different dimensions.

Two dice subsets are created:
- Transactions where **Season = Winter and Category = Clothing**
- Transactions where **Season = Summer and Category = Footwear**

This helps analyze more specific combinations of attributes within the dataset.

In [13]:
dice_data = df[
    (df["Season"] == "Winter") &
    (df["Category"] == "Clothing")
]

print("Dice Result (Winter Clothing Purchases):")
print(dice_data.head())

Dice Result (Winter Clothing Purchases):
    Customer ID  Age Gender Item Purchased  Category  Purchase Amount (USD)  \
0             1   55   Male         Blouse  Clothing                     53   
1             2   19   Male        Sweater  Clothing                     64   
7             8   27   Male         Shorts  Clothing                     34   
11           12   30   Male         Shorts  Clothing                     68   
15           16   64   Male          Skirt  Clothing                     81   

        Location Size     Color  Season  Review Rating Subscription Status  \
0       Kentucky    L      Gray  Winter            3.1                 Yes   
1          Maine    L    Maroon  Winter            3.1                 Yes   
7      Louisiana    L  Charcoal  Winter            3.2                 Yes   
11        Hawaii    S     Olive  Winter            4.9                 Yes   
15  Rhode Island    M      Teal  Winter            2.8                 Yes   

    Shipping Ty

In [14]:
dice_data = df[
    (df["Season"] == "Summer") &
    (df["Category"] == "Footwear")
]

print("Dice Result (Summer Footwear Purchases):")
print(dice_data.head())

Dice Result (Summer Footwear Purchases):
     Customer ID  Age Gender Item Purchased  Category  Purchase Amount (USD)  \
5              6   46   Male       Sneakers  Footwear                     20   
80            81   19   Male        Sandals  Footwear                     72   
81            82   67   Male          Shoes  Footwear                     96   
117          118   50   Male        Sandals  Footwear                     32   
147          148   63   Male          Shoes  Footwear                     64   

         Location Size   Color  Season  Review Rating Subscription Status  \
5         Wyoming    M   White  Summer            2.9                 Yes   
80       New York   XL    Blue  Summer            3.3                 Yes   
81       Virginia    L  Maroon  Summer            2.6                 Yes   
117  South Dakota    L  Yellow  Summer            4.6                 Yes   
147         Idaho    M  Purple  Summer            3.5                 Yes   

      Shipping 

In [15]:
dice_summary = dice_data.groupby("Item Purchased")["Purchase Amount (USD)"].sum()
print(dice_summary)

Item Purchased
Boots       2375
Sandals     2123
Shoes       2781
Sneakers    2114
Name: Purchase Amount (USD), dtype: int64


In [16]:
print("\n===== DATA CUBE =====")
print(data_cube)

print("\n===== ROLL-UP =====")
print(rollup)

print("\n===== DRILL-DOWN =====")
print(drilldown)

print("\n===== SLICE (Winter Purchases) =====")
print(slice_summary)

print("\n===== DICE (Winter Clothing Purchases) =====")
print(dice_summary)


===== DATA CUBE =====
Season        Fall  Spring  Summer  Winter
Category                                  
Accessories  19874   17007   19028   18291
Clothing     26220   27692   23078   27274
Footwear      8665    9555    9393    8480
Outerwear     5259    4425    4278    4562

===== ROLL-UP =====
Category
Accessories     74200
Clothing       104264
Footwear        36093
Outerwear       18524
Name: Purchase Amount (USD), dtype: int64

===== DRILL-DOWN =====
Category     Item Purchased
Accessories  Backpack           8636
             Belt               9635
             Gloves             8477
             Handbag            8857
             Hat                9375
             Jewelry           10010
             Scarf              9561
             Sunglasses         9649
Clothing     Blouse            10410
             Dress             10320
             Hoodie             8767
             Jeans              7548
             Pants             10090
             Shirt        